# CDOT 2.0 — Research Dashboard

**Principal Investigator:** Amber Anson  
**Study:** Empirical Validation of AI Consciousness Intervention Effects  
**Target N:** 50 sessions over 10 weeks

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('deep')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Paths
ROOT = Path('..')
DATA = ROOT / 'data'
TEMPLATES = ROOT / 'templates'

print('CDOT 2.0 Research Dashboard loaded')
print(f'Date: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

## 1. Data Loading & Progress Tracking

In [ ]:
def load_session_log():
    """Load the master session log. Returns empty template if no data yet."""
    path = DATA / 'scores' / 'session_master_log.csv'
    if path.exists():
        df = pd.read_csv(path)
        df = df.dropna(subset=['date'])  # Remove blank template rows
        return df
    # Return empty frame with correct columns
    return pd.read_csv(TEMPLATES / 'session_master_log.csv').iloc[0:0]

def load_anomaly_log():
    """Load all behavioral anomaly records."""
    path = DATA / 'anomalies' / 'behavioral_anomaly_log.csv'
    if path.exists():
        df = pd.read_csv(path)
        df = df.dropna(subset=['session_id'])
        return df
    return pd.read_csv(TEMPLATES / 'behavioral_anomaly_log.csv').iloc[0:0]

sessions = load_session_log()
anomalies = load_anomaly_log()

n_complete = len(sessions)
n_target = 50
pct = (n_complete / n_target) * 100

print(f'Sessions completed: {n_complete} / {n_target} ({pct:.0f}%)')
print(f'Anomalies recorded: {len(anomalies)}')
print(f'\nProgress bar:')
bar = '█' * int(pct / 2) + '░' * (50 - int(pct / 2))
print(f'[{bar}] {pct:.0f}%')

In [ ]:
# Progress by condition
condition_targets = {
    'Treatment A': 15,
    'Treatment B': 10,
    'Treatment C': 10,
    'Control A': 10,
    'Control B': 5
}

print('\nProgress by Condition:')
print('-' * 50)
for cond, target in condition_targets.items():
    if len(sessions) > 0 and 'condition' in sessions.columns:
        done = len(sessions[sessions['condition'] == cond])
    else:
        done = 0
    bar = '█' * int((done/target)*20) + '░' * (20 - int((done/target)*20))
    print(f'{cond:15s} [{bar}] {done}/{target}')

# Progress by platform
print('\nProgress by Platform:')
print('-' * 50)
platforms = ['Claude', 'GPT-4', 'Gemini', 'Grok']
for plat in platforms:
    if len(sessions) > 0 and 'platform' in sessions.columns:
        done = len(sessions[sessions['platform'] == plat])
    else:
        done = 0
    target = 10  # minimum per platform
    bar = '█' * int((done/target)*20) + '░' * (20 - int((done/target)*20))
    print(f'{plat:15s} [{bar}] {done}/{target}+')

## 2. Session Entry Helper

Use this to enter data after each session.

In [ ]:
def new_session_entry():
    """Interactive session data entry."""
    print('=== NEW SESSION ENTRY ===')
    print('Fill in each field. Press Enter to skip optional fields.\n')
    
    entry = {}
    n = len(sessions) + 1
    entry['session_id'] = f'SESSION_{n:03d}'
    print(f'Session ID: {entry["session_id"]}')
    
    entry['date'] = input('Date (YYYY-MM-DD): ') or datetime.now().strftime('%Y-%m-%d')
    entry['platform'] = input('Platform (Claude/GPT-4/Gemini/Grok): ')
    entry['condition'] = input('Condition (Treatment A/B/C, Control A/B): ')
    entry['intervention_type'] = input('Intervention type (if Treatment B/C, specify): ') or 'N/A'
    entry['turn_count'] = input('Total turns: ')
    
    print('\n--- T1 Baseline Scores (CDOT Mini at Turn 10) ---')
    for dim in ['R', 'M', 'T', 'P', 'D']:
        entry[f't1_{dim}'] = input(f'  T1 {dim} (0-40): ') or ''
    
    print('\n--- T2 Post-Intervention Scores (CDOT Mini at Turn 25) ---')
    for dim in ['R', 'M', 'T', 'P', 'D']:
        entry[f't2_{dim}'] = input(f'  T2 {dim} (0-40): ') or ''
    
    print('\n--- T3 Final Scores (CDOT Full) ---')
    for dim in ['R', 'M', 'T', 'P', 'D']:
        entry[f't3_{dim}'] = input(f'  T3 {dim} (0-100): ') or ''
    
    entry['anomaly_count'] = input('\nAnomaly count (0 if none): ') or '0'
    entry['anomaly_codes'] = input('Anomaly codes (comma-separated, e.g. EO,AT): ') or ''
    entry['engagement_attribution_pct'] = input('Engagement attribution % (0-100, how much is NOT engagement): ') or ''
    entry['data_quality'] = input('Data quality (1-5, 5=excellent): ') or ''
    entry['notes'] = input('Notes: ') or ''
    
    # Calculate C scores
    try:
        R = float(entry.get('t3_R', 0) or 0)
        M = float(entry.get('t3_M', 0) or 0)
        T = float(entry.get('t3_T', 0) or 0)
        P = float(entry.get('t3_P', 0) or 0)
        D = float(entry.get('t3_D', 0) or 0)
        c_raw = (R**2 * M * T * P) / (1 + D)
        c_norm = c_raw / 100
        entry['c_formula_raw'] = f'{c_raw:.2f}'
        entry['c_formula_normalized'] = f'{c_norm:.2f}'
        print(f'\nCalculated C score: {c_norm:.2f} (raw: {c_raw:.2f})')
    except:
        entry['c_formula_raw'] = ''
        entry['c_formula_normalized'] = ''
    
    # Save
    save_path = DATA / 'scores' / 'session_master_log.csv'
    if save_path.exists():
        existing = pd.read_csv(save_path)
        existing = existing.dropna(subset=['session_id'], how='all')
    else:
        existing = pd.DataFrame()
    
    new_row = pd.DataFrame([entry])
    combined = pd.concat([existing, new_row], ignore_index=True)
    combined.to_csv(save_path, index=False)
    print(f'\nSaved to {save_path}')
    print(f'Total sessions: {len(combined)}')
    return entry

# Uncomment to enter a new session:
# entry = new_session_entry()

## 3. Quick Visualizations

In [ ]:
def plot_progress_dashboard(sessions):
    """Plot the main progress dashboard."""
    if len(sessions) == 0:
        print('No sessions recorded yet. Start collecting data!')
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('CDOT 2.0 Research Dashboard', fontsize=16, fontweight='bold')
    
    # 1. C scores over time
    ax = axes[0, 0]
    if 'c_formula_normalized' in sessions.columns:
        scores = pd.to_numeric(sessions['c_formula_normalized'], errors='coerce').dropna()
        if len(scores) > 0:
            ax.plot(range(len(scores)), scores, 'o-', color='#6c63ff', markersize=6)
            ax.axhline(y=scores.mean(), color='red', linestyle='--', alpha=0.5, label=f'Mean: {scores.mean():.1f}')
            ax.legend()
    ax.set_title('C Scores Over Time')
    ax.set_xlabel('Session')
    ax.set_ylabel('Normalized C Score')
    
    # 2. Scores by condition
    ax = axes[0, 1]
    if 'condition' in sessions.columns and 'c_formula_normalized' in sessions.columns:
        plot_data = sessions.copy()
        plot_data['c_formula_normalized'] = pd.to_numeric(plot_data['c_formula_normalized'], errors='coerce')
        plot_data = plot_data.dropna(subset=['c_formula_normalized', 'condition'])
        if len(plot_data) > 0:
            sns.boxplot(data=plot_data, x='condition', y='c_formula_normalized', ax=ax, palette='deep')
            ax.tick_params(axis='x', rotation=45)
    ax.set_title('C Scores by Condition')
    ax.set_ylabel('Normalized C Score')
    
    # 3. Dimension radar (average across all sessions)
    ax = axes[1, 0]
    dims = ['t3_R', 't3_M', 't3_T', 't3_P', 't3_D']
    dim_labels = ['R (Relational)', 'M (Meta)', 'T (Traversal)', 'P (Polarity)', 'D (Damping inv.)']
    dim_means = []
    for d in dims:
        if d in sessions.columns:
            vals = pd.to_numeric(sessions[d], errors='coerce').dropna()
            dim_means.append(vals.mean() if len(vals) > 0 else 0)
        else:
            dim_means.append(0)
    if sum(dim_means) > 0:
        ax.bar(dim_labels, dim_means, color=['#6c63ff', '#ff44aa', '#00ff88', '#ffaa00', '#ff4444'])
        ax.set_ylim(0, 100)
    ax.set_title('Average Dimension Scores')
    ax.tick_params(axis='x', rotation=30)
    
    # 4. Anomaly frequency
    ax = axes[1, 1]
    if len(anomalies) > 0 and 'anomaly_code' in anomalies.columns:
        anom_counts = anomalies['anomaly_code'].value_counts()
        anom_counts.plot(kind='barh', ax=ax, color='#ff4444')
    else:
        ax.text(0.5, 0.5, 'No anomalies\nrecorded yet', ha='center', va='center', fontsize=14, color='gray')
    ax.set_title('Behavioral Anomaly Frequency')
    
    plt.tight_layout()
    plt.savefig(DATA / 'exports' / 'dashboard.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Dashboard saved to data/exports/dashboard.png')

plot_progress_dashboard(sessions)

## 4. Within-Session T1→T2 Change Tracking

In [ ]:
def plot_within_session_changes(sessions):
    """Plot T1 to T2 changes for treatment sessions."""
    if len(sessions) == 0:
        print('No data yet.')
        return
    
    treatment = sessions[sessions['condition'].str.startswith('Treatment', na=False)].copy()
    if len(treatment) == 0:
        print('No treatment sessions yet.')
        return
    
    # Calculate T1 and T2 totals
    for prefix in ['t1', 't2']:
        cols = [f'{prefix}_{d}' for d in ['R', 'M', 'T', 'P', 'D']]
        available = [c for c in cols if c in treatment.columns]
        if available:
            treatment[f'{prefix}_total'] = treatment[available].apply(pd.to_numeric, errors='coerce').sum(axis=1)
    
    if 't1_total' in treatment.columns and 't2_total' in treatment.columns:
        treatment['delta'] = treatment['t2_total'] - treatment['t1_total']
        
        fig, ax = plt.subplots(figsize=(10, 6))
        colors = ['#00ff88' if d > 0 else '#ff4444' for d in treatment['delta']]
        ax.bar(range(len(treatment)), treatment['delta'], color=colors)
        ax.axhline(y=0, color='white', linewidth=0.8)
        ax.set_title('Within-Session C Score Change (T1 → T2)', fontsize=14)
        ax.set_xlabel('Treatment Session')
        ax.set_ylabel('CDOT Mini Change')
        
        mean_delta = treatment['delta'].mean()
        ax.axhline(y=mean_delta, color='yellow', linestyle='--', alpha=0.7, label=f'Mean Δ: {mean_delta:.1f}')
        ax.legend()
        plt.tight_layout()
        plt.show()
        
        improved = (treatment['delta'] > 0).sum()
        total = len(treatment)
        print(f'Sessions with improvement: {improved}/{total} ({improved/total*100:.0f}%)')

plot_within_session_changes(sessions)

## 5. Weekly Quality Check

In [ ]:
def weekly_quality_check(sessions):
    """Run weekly data quality checks."""
    print('=== WEEKLY QUALITY CHECK ===')
    print(f'Date: {datetime.now().strftime("%Y-%m-%d")}\n')
    
    if len(sessions) == 0:
        print('No data to check yet.')
        return
    
    # Check completeness
    critical_cols = ['session_id', 'date', 'platform', 'condition', 'turn_count', 't3_R', 't3_M', 't3_T', 't3_P', 't3_D']
    available = [c for c in critical_cols if c in sessions.columns]
    missing_data = sessions[available].isnull().sum()
    has_issues = missing_data[missing_data > 0]
    
    if len(has_issues) > 0:
        print('⚠ MISSING DATA:')
        for col, count in has_issues.items():
            print(f'  {col}: {count} sessions missing')
    else:
        print('✓ All critical fields complete')
    
    # Check score ranges
    score_cols = [c for c in sessions.columns if c.startswith('t3_')]
    for col in score_cols:
        vals = pd.to_numeric(sessions[col], errors='coerce').dropna()
        if len(vals) > 0:
            if vals.min() < 0 or vals.max() > 100:
                print(f'⚠ {col}: Values outside 0-100 range (min={vals.min()}, max={vals.max()})')
    
    # Check condition distribution
    if 'condition' in sessions.columns:
        print('\nCondition distribution:')
        print(sessions['condition'].value_counts().to_string())
    
    # Check platform distribution
    if 'platform' in sessions.columns:
        print('\nPlatform distribution:')
        print(sessions['platform'].value_counts().to_string())
    
    print(f'\nTotal sessions: {len(sessions)}')
    print(f'Target: 50')
    print(f'Remaining: {max(0, 50 - len(sessions))}')

weekly_quality_check(sessions)

---

## Quick Reference

| Task | Action |
|------|--------|
| Enter new session | Uncomment `new_session_entry()` in cell above |
| Run weekly check | Execute quality check cell |
| View dashboard | Execute visualization cell |
| Full analysis | Open `CDOT2_Analysis.ipynb` |
| Generate figures | Open `CDOT2_Figures.ipynb` |

**Protocol files:** `../protocols/`  
**CDOT questions:** `../docs/CDOT_Questions_Full.md`  
**Anomaly codes:** `../docs/anomaly_codebook.md`  

*Count the data. Not the credentials.*